# 2.5D Spiking Autoregressive MRI Reconstruction (Spike-AR vs. Dense Baselines)

This notebook evaluates **Paradigm B** (multi-step Parametric Leaky Integrate-and-Fire with direct analog drive and non-spiking membrane readout) against dense convolutional baselines:
1. `vanilla_unet`: Dense U-Net, unmasked $\mathcal{L}_1$ loss (unbounded spatial drift control).
2. `baseline_unet`: Dense U-Net, shape-mask concatenated, spatial $\mathcal{L}_1$ loss (textural dragging & blur control).
3. `spike_unet_t4`: Spiking U-Net under Paradigm B ($T_{\text{sim}} = 4$).
4. `spike_unet_t2`: Spiking U-Net under Paradigm B ($T_{\text{sim}} = 2$).

### 1. Environment Setup and Dependency Installation

In [ ]:
import os
import sys

# Detect Kaggle environment
KAGGLE = bool(os.environ.get("KAGGLE_URL_BASE") or os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))

if KAGGLE:
    print(">>> Running on Kaggle: Installing spirec from develop branch...")
    # Clean install from the repository develop branch
    %pip uninstall -q --yes spirec
    %pip install --upgrade "spirec @ git+https://github.com/CV-RL-Deep/MRI-Spiking-Reconstruction.git@develop"
    %pip install -q yacs einops
else:
    print(">>> Running in local/cluster environment. Ensuring local spirec is in sys.path...")
    sys.path.insert(0, os.path.abspath(".."))

In [ ]:
import copy
import glob
import random
import shutil
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from spirec.configs.config import CFG, Config
from spirec.core.env import ENV
from spirec.core.utils import logger, set_global_seeds, PipelineTimer
from spirec.data.cache import ActiveLoader, StaticLoader, VolumeLoader
from spirec.data.dataset import (
    AutoregressiveSliceDataset,
    SequentialValidationDataset,
    collate_autoregressive,
)
from spirec.data.files import find_t1_files, get_brats_subjects
from spirec.evaluation.frechet import FrechetEvaluator
from spirec.evaluation.metrics import calculate_step_metrics
from spirec.evaluation.visualizer import VisualizationSuite
from spirec.inference.rollout import VolumeReconstructor
from spirec.models import ModelBuilder
from spirec.training.trainer import Trainer

# Verify Hardware Acceleration
set_global_seeds(42, deterministic=False)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Compute Engine: {device}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    print(f"Allocated VRAM: {torch.cuda.memory_allocated() / (1024**2):.2f} MB")

### 2. Dataset Discovery and Adaptive Configuration

In [ ]:
# Identify execution mode: Interactive (smoke test) vs. Batch (production run)
is_interactive = os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "Interactive") == "Interactive"

# Runtime adjustments
CFG.train.epochs = 5 if is_interactive else 60
CFG.train.batch_size = 8 if is_interactive else 16
CFG.data.padded_size = (160, 160)
num_eval_vols = 2 if is_interactive else 10
rollout_span = 5 if is_interactive else 20  # Total steps k = 10 vs 40

print(f"Run Profile: {'Interactive (Fast Test)' if is_interactive else 'Batch (Full Run)'}")
print(f"Epochs per model: {CFG.train.epochs} | Rollout Span: +/- {rollout_span} slices")

# File discovery
ixi_files = find_t1_files(CFG.data.data_root_ixi)
brats_subjects = get_brats_subjects(CFG.data.data_root_brats)

if not ixi_files:
    raise FileNotFoundError(f"No IXI T1 files located at: {CFG.data.data_root_ixi}")

# Enforce deterministic splits
random.seed(42)
random.shuffle(ixi_files)
ixi_split = int(0.9 * len(ixi_files))
train_ixi, val_ixi = ixi_files[:ixi_split], ixi_files[ixi_split:]

random.shuffle(brats_subjects)
brats_split = int(0.9 * len(brats_subjects))
train_brats, val_brats = brats_subjects[:brats_split], brats_subjects[brats_split:]

eval_files = val_ixi[:num_eval_vols]

print(f"Data Distribution -> Train IXI: {len(train_ixi)} | Val IXI: {len(val_ixi)}")
print(f"Data Distribution -> Train BraTS: {len(train_brats)} | Val BraTS: {len(val_brats)}")
print(f"Evaluation Targets Locked: {len(eval_files)} volumes")

### 3. Defining the 4-Model Experimental Matrix

In [ ]:
def build_experiment_matrix(base_cfg: Config) -> dict:
    matrix = {}

    # Variant 1: Dense Vanilla U-Net (Unbounded spatial drift control)
    cfg_vanilla = copy.deepcopy(base_cfg)
    cfg_vanilla.model.name = "vanilla_unet"
    cfg_vanilla.model.architecture = "vanilla_unet"
    cfg_vanilla.data.input_last_target_mask = False
    matrix["vanilla_unet"] = cfg_vanilla

    # Variant 2: Dense Baseline U-Net (Target mask + Spatial L1 -> Textural blur control)
    cfg_baseline = copy.deepcopy(base_cfg)
    cfg_baseline.model.name = "baseline_unet"
    cfg_baseline.model.architecture = "baseline_unet"
    cfg_baseline.data.input_last_target_mask = True
    matrix["baseline_unet"] = cfg_baseline

    # Variant 3: Spike-AR-UNet (Paradigm B, T_sim = 4)
    cfg_snn_t4 = copy.deepcopy(base_cfg)
    cfg_snn_t4.model.name = "spike_unet_t4"
    cfg_snn_t4.model.architecture = "spike_unet"
    cfg_snn_t4.data.input_last_target_mask = False
    cfg_snn_t4.snn.t_sim = 4
    matrix["spike_unet_t4"] = cfg_snn_t4

    # Variant 4: Spike-AR-UNet (Paradigm B, T_sim = 2)
    cfg_snn_t2 = copy.deepcopy(base_cfg)
    cfg_snn_t2.model.name = "spike_unet_t2"
    cfg_snn_t2.model.architecture = "spike_unet"
    cfg_snn_t2.data.input_last_target_mask = False
    cfg_snn_t2.snn.t_sim = 2
    matrix["spike_unet_t2"] = cfg_snn_t2

    return matrix

experiments = build_experiment_matrix(CFG)
print(f"Configured {len(experiments)} experimental tracks: {list(experiments.keys())}")

### 4. Multi-Model Training and Autoregressive Rollouts

In [ ]:
os.makedirs(CFG.data.results_dir, exist_ok=True)
os.makedirs(CFG.data.weights_dir, exist_ok=True)

# 1. Setup Static Validation Memory Pool
val_loader_mem = StaticLoader(CFG, val_ixi[:15], val_brats[:15])
val_dataset = SequentialValidationDataset(val_loader_mem, CFG, slices_per_vol=10)
val_dataloader = DataLoader(
    val_dataset,
    batch_size=CFG.train.batch_size,
    shuffle=False,
    collate_fn=lambda b: collate_autoregressive(b, target_spatial_size=CFG.data.padded_size),
)

all_rollout_metrics = []

for ab_name, config in experiments.items():
    print("\n" + "=" * 65)
    print(f"   STARTING PHASE: {ab_name.upper()}")
    print("=" * 65)
    
    weights_path = os.path.join(config.data.weights_dir, f"{config.model.name}_best.pt")
    
    # Active loader manages rolling memory cache during training
    with ActiveLoader(config, train_ixi, train_brats) as train_loader_mem:
        model = ModelBuilder.build(config).to(device)
        
        if os.path.exists(weights_path):
            print(f"[*] Found pre-trained weights for {ab_name}. Loading: {weights_path}")
            checkpoint = torch.load(weights_path, map_location=device)
            model.load_state_dict(checkpoint["model_state_dict"])
        else:
            print(f"[*] Training {ab_name} from scratch...")
            train_dataset = AutoregressiveSliceDataset(
                train_loader_mem, config, samples_per_epoch=200 if is_interactive else 3000
            )
            train_dataloader = DataLoader(
                train_dataset,
                batch_size=config.train.batch_size,
                shuffle=True,
                collate_fn=lambda b: collate_autoregressive(b, target_spatial_size=config.data.padded_size),
            )
            
            trainer = Trainer(config, model, train_dataloader, val_dataloader, device=device)
            with PipelineTimer(f"Training [{ab_name}]"):
                trainer.fit(num_epochs=config.train.epochs)
                
            # Reload optimal checkpoint
            if os.path.exists(weights_path):
                checkpoint = torch.load(weights_path, map_location=device)
                model.load_state_dict(checkpoint["model_state_dict"])

        # Unconstrained Autoregressive Inference (Longitudinal Rollout)
        print(f"\n[*] Executing unconstrained rollouts across {len(eval_files)} evaluation volumes...")
        reconstructor = VolumeReconstructor(model, config, device=device)
        
        for vol_path in tqdm(eval_files, desc=f"Rollouts [{ab_name}]"):
            vol_id = os.path.basename(vol_path).replace(".nii.gz", "").replace(".nii", "")
            vol_obj = next((v for v in val_loader_mem.pool.get("ixi", []) if v.path == vol_path), None)
            
            if vol_obj is None:
                vol_obj = VolumeLoader.load(vol_path)
            if vol_obj is None:
                continue

            volume = vol_obj.t1
            center = volume.shape[0] // 2
            start_z = max(0, center - rollout_span)
            end_z = min(volume.shape[0], center + rollout_span)

            # Synthesize contiguous block via pure forward autoregression
            recon_vol = reconstructor.autoregressive_restore(
                volume=volume,
                start_idx=start_z,
                end_idx=end_z,
                direction="forward",
                mask_volume=None,
                disable_pbar=True,
            )

            # Track step-by-step metric degradation
            metrics = calculate_step_metrics(
                gt_vol=volume,
                pred_vol=recon_vol,
                start_idx=start_z,
                end_idx=end_z,
                direction="forward",
                ablation_name=ab_name,
                vol_id=vol_id,
            )
            all_rollout_metrics.extend(metrics)

            # Save arrays for subsequent Fréchet and visual inspection
            np.save(os.path.join(config.data.results_dir, f"{ab_name}_{vol_id}.npy"), recon_vol)
            np.save(os.path.join(config.data.results_dir, f"gt_{vol_id}.npy"), volume)

# Export metrics
df_metrics = pd.DataFrame(all_rollout_metrics)
metrics_csv_path = os.path.join(CFG.data.results_dir, "ablation_metrics.csv")
df_metrics.to_csv(metrics_csv_path, index=False)
print(f"\n[+] Rollout degradation metrics saved to: {metrics_csv_path}")

### 5. Quantitative Drift Analysis and Error Curves

In [ ]:
df_metrics = pd.read_csv(metrics_csv_path)

# Plotting aesthetics
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 9,
    "axes.labelsize": 10,
    "legend.fontsize": 8,
    "figure.titlesize": 11,
})

palette = {
    "vanilla_unet": "#94a3b8",   # Slate Gray (Uncontrolled Drift)
    "baseline_unet": "#e63946",  # Red (Textural Blur)
    "spike_unet_t4": "#1d3557",  # Deep Navy (Spike-AR T=4)
    "spike_unet_t2": "#2a9d8f",  # Teal (Spike-AR T=2)
}

labels = {
    "vanilla_unet": "Dense Vanilla U-Net (Unbounded Drift)",
    "baseline_unet": "Dense Baseline U-Net (Masked)",
    "spike_unet_t4": "Spike-AR-UNet (Paradigm B, T=4)",
    "spike_unet_t2": "Spike-AR-UNet (Paradigm B, T=2)",
}

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

# 1. Structural Similarity (SSIM) Decay
sns.lineplot(
    data=df_metrics, x="Rollout_Step_k", y="SSIM", hue="Ablation",
    palette=palette, ax=axes[0], linewidth=1.8
)
axes[0].set_title("Structural Degradation (SSIM)")
axes[0].set_xlabel("Autoregressive Step (k)")
axes[0].set_ylabel("SSIM (Higher is Better)")
axes[0].grid(True, linestyle="--", alpha=0.5)

# 2. Focal Frequency Loss (FFL) Explosion
sns.lineplot(
    data=df_metrics, x="Rollout_Step_k", y="FFL", hue="Ablation",
    palette=palette, ax=axes[1], linewidth=1.8
)
axes[1].set_title("Frequency Degradation (FFL)")
axes[1].set_xlabel("Autoregressive Step (k)")
axes[1].set_ylabel("Focal Frequency Error (Lower is Better)")
axes[1].grid(True, linestyle="--", alpha=0.5)

# 3. Spatial Gradient Error
sns.lineplot(
    data=df_metrics, x="Rollout_Step_k", y="Grad_Error", hue="Ablation",
    palette=palette, ax=axes[2], linewidth=1.8
)
axes[2].set_title("Edge Sharpness Attenuation")
axes[2].set_xlabel("Autoregressive Step (k)")
axes[2].set_ylabel("Gradient Error (Lower is Better)")
axes[2].grid(True, linestyle="--", alpha=0.5)

for ax in axes:
    handles, lbls = ax.get_legend_handles_labels()
    ax.legend(handles=handles, labels=[labels.get(l, l) for l in lbls], loc="best", framealpha=0.8)

plt.tight_layout()
fig_save_path = os.path.join(CFG.data.figures_dir, "quantitative_drift_curves.pdf")
os.makedirs(CFG.data.figures_dir, exist_ok=True)
plt.savefig(fig_save_path, dpi=300, bbox_inches="tight")
plt.show()

# Macro Performance Table at Horizon Limit
max_k = df_metrics["Rollout_Step_k"].max()
summary_table = (
    df_metrics[df_metrics["Rollout_Step_k"] == max_k]
    .groupby("Ablation")[["SSIM", "PSNR", "MAE", "Grad_Error", "FFL"]]
    .mean()
    .reset_index()
)
print(f"\n--- Quantitative Performance at Step k={max_k} ---")
display(summary_table)

### 6. Qualitative Slice Matrix Comparison

In [ ]:
target_vol_id = os.path.basename(eval_files[0]).replace(".nii.gz", "").replace(".nii", "")
gt_path = os.path.join(CFG.data.results_dir, f"gt_{target_vol_id}.npy")

if os.path.exists(gt_path):
    gt_vol = np.load(gt_path)
    models_to_render = ["vanilla_unet", "baseline_unet", "spike_unet_t4", "spike_unet_t2"]
    
    # Extract temporal indices across rollout
    center = gt_vol.shape[0] // 2
    step_indices = [center - rollout_span + 1, center, center + rollout_span - 2]
    
    fig, axes = plt.subplots(
        len(models_to_render) + 1, len(step_indices),
        figsize=(3.2 * len(step_indices), 2.8 * (len(models_to_render) + 1)),
        gridspec_kw={"wspace": 0.05, "hspace": 0.15}
    )

    # Render Ground Truth Row
    for col_idx, z_slice in enumerate(step_indices):
        axes[0, col_idx].imshow(gt_vol[z_slice], cmap="bone", vmin=0, vmax=1)
        axes[0, col_idx].set_title(f"Step z={z_slice} (GT)", fontsize=9)
        axes[0, col_idx].axis("off")
    axes[0, 0].set_ylabel("Ground Truth", fontsize=9, weight="bold")

    # Render Model Prediction Rows
    for row_idx, ab in enumerate(models_to_render, start=1):
        pred_path = os.path.join(CFG.data.results_dir, f"{ab}_{target_vol_id}.npy")
        if os.path.exists(pred_path):
            pred_vol = np.load(pred_path)
            for col_idx, z_slice in enumerate(step_indices):
                axes[row_idx, col_idx].imshow(pred_vol[z_slice], cmap="bone", vmin=0, vmax=1)
                axes[row_idx, col_idx].axis("off")
        axes[row_idx, 0].set_ylabel(labels.get(ab, ab), fontsize=8, weight="bold")

    plt.suptitle(f"Longitudinal Autoregressive Progression ({target_vol_id})", fontsize=11, weight="bold")
    plt.show()

### 7. Fréchet Video Distance (FVD) & 3D-FID Benchmarking

In [ ]:
print("[*] Computing 3D Structural Realism (3D-FID) and Sequential Continuity (FVD)...")
evaluator_frechet = FrechetEvaluator(CFG, device=device)
df_frechet = evaluator_frechet.evaluate(list(experiments.keys()))

if not df_frechet.empty:
    print("\n--- Fréchet Distributional Metric Results ---")
    display(df_frechet)
    
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
    sns.barplot(data=df_frechet, x="Ablation", y="3D-FID", palette=palette, ax=axes[0])
    axes[0].set_title("3D Structural Realism (3D-FID ↓)")
    axes[0].set_xticklabels([labels.get(t.get_text(), t.get_text()) for t in axes[0].get_xticklabels()], rotation=35, ha="right")
    axes[0].grid(True, axis="y", linestyle="--", alpha=0.5)

    sns.barplot(data=df_frechet, x="Ablation", y="FVD", palette=palette, ax=axes[1])
    axes[1].set_title("Z-Axis Sequential Continuity (FVD ↓)")
    axes[1].set_xticklabels([labels.get(t.get_text(), t.get_text()) for t in axes[1].get_xticklabels()], rotation=35, ha="right")
    axes[1].grid(True, axis="y", linestyle="--", alpha=0.5)

    plt.tight_layout()
    plt.savefig(os.path.join(CFG.data.figures_dir, "frechet_metrics_barplot.pdf"), dpi=300, bbox_inches="tight")
    plt.show()

### 8. Artifact Packaging for Export

In [ ]:
archive_name = "spirec_ablation_artifacts"
print(f"[*] Packaging evaluation results and checkpoints into {archive_name}.zip...")

# Bundle results, weights, and figures
!zip -q -r -9 {archive_name}.zip ablation_results paper_figures checkpoints memory_telemetry.csv 2>/dev/null || true

if os.path.exists(f"{archive_name}.zip"):
    size_mb = os.path.getsize(f"{archive_name}.zip") / (1024**2)
    print(f"[+] Archive ready: {archive_name}.zip ({size_mb:.2f} MB)")